# Station 2: Log trace — 15 points

Reconstruct a missing page incident from synthetic local logs. Identify what the evidence establishes and what remains unknown.

**Scope:** These are practice scenarios inferred from `urtuu.md`, not official competition tasks.

**Offline setup:** Python 3 kernel; all exercise code uses the standard library. Run cells from top to bottom. No downloads or internet are required.

**How to study:** Read the worked example, then attempt the three exercises before opening the solution section. Use a 30–40 minute timer. Record any recovered fragment exactly.

## Learn it visually



**How to read this:** Read left to right. GET is an action, `/rules` is the resource, and 200/403/404 describe outcomes. A log records these exchanges so you can reconstruct the sequence later.

**Try before continuing:** Does a logged DELETE with status 403 prove that the page was deleted?

<details><summary>Show the answer</summary>

No. The server refused the request. Look for a successful change and corroborating evidence.

</details>


## Mini-course: reading a web incident like a timeline

**What the guideline confirms:** A page is missing; use logs to trace the responsible actor. The log format and available tools are unspecified.

### Lesson 1 — Requests and responses
A browser is a client. It sends a request to a server, which responds. A request contains a method (action) and a path (resource). `GET /rules` asks to read the rules. POST often submits data; DELETE requests removal. Actual behavior depends on the application.

The response status summarizes the result: 200 means success, 204 means success with no response body, 403 means access refused, 404 means resource not found, and 500 means a server error. A request to delete something is not proof that deletion succeeded.

### Lesson 2 — Read each field
A timestamp says when an event was logged. An IP address identifies a network endpoint. A username identifies an account. A request ID links records belonging to one operation. A path identifies the requested resource. Keep these separate: an IP or account is not automatically a unique person.

### Lesson 3 — Reconstruct cause and effect
Start with the missing page, filter logs to that path, and sort by time. Find its last successful read and first failed read. Investigate changes between those events. Cross-check an audit log, which records actions such as deletion or permission changes. An unrelated 404 for an icon is usually weaker evidence than a successful deletion of the exact page.

| Possible task (prediction) | First action |
|---|---|
| Find who deleted a page | Identify a successful change and correlate account/request ID |
| Spot a suspicious login | Compare failures, successes, time, and source |
| Recover a hidden clue in a URL | Inspect path and query text; consider URL encoding |
| Many noisy log lines | Filter by resource and time before counting events |

**Learn first:** methods, common statuses, filtering, sorting, and evidence-based explanations. **Extra practice:** query parameters (`?id=7`), URL encoding (`%20` is a space), session IDs, and time zones.


## Worked example: status is evidence
A `DELETE /page` returning **403** was denied. A later `DELETE /page` returning **204** indicates a successful request under this mock service’s documented behavior. A subsequent **404** supports that the page is now missing. An IP identifies a network source, not necessarily a person.

For these exercises, assume synchronized clocks, complete logs, and a mock service where successful DELETE removes the page. Real systems require checking those assumptions.

In [ ]:
logs = [('10:00:01', '10.0.0.11', 'naraa', 'GET', '/home', 200), ('10:00:05', '10.0.0.12', 'bold', 'GET', '/rules', 200), ('10:01:00', '10.0.0.23', 'guest', 'DELETE', '/rules', 403), ('10:01:10', '10.0.0.44', 'admin', 'POST', '/login', 200), ('10:01:14', '10.0.0.44', 'admin', 'DELETE', '/rules', 204), ('10:01:15', '10.0.0.12', 'bold', 'GET', '/rules', 404), ('10:01:16', '10.0.0.44', 'admin', 'GET', '/rules', 404), ('10:02:00', '10.0.0.23', 'guest', 'GET', '/favicon.ico', 404)]
columns = ("time", "ip", "user", "method", "path", "status")
records = [dict(zip(columns, row)) for row in logs]
for r in records:
    print("{time} {ip:12} {user:6} {method:6} {path:14} {status}".format(**r))

## Your exercises

1. **Trace (8 minutes):** Which request most directly explains the missing `/rules` page? Give timestamp, IP, account, method, and status. Why is the guest’s request insufficient?
2. **Filter (8 minutes):** List only `/rules` events in chronological order. Count 404 responses by IP across all paths, then only for `/rules`. Is the IP with most 404s necessarily responsible?
3. **Cross-check (10 minutes):** Use the audit records below to find the request ID and recovered fragment for the successful deletion. Distinguish an account from the human operating it. What additional evidence would you request?

In [ ]:
audit = [
    {"time":"10:01:00", "ip":"10.0.0.23", "request_id":"r17", "action":"delete_denied", "fragment":None},
    {"time":"10:01:14", "ip":"10.0.0.44", "request_id":"r19", "action":"delete_page", "fragment":"TRACE"},
]
# Your analysis here

---
# Solutions — stop here until you have attempted the exercises
Compare both your answer and your reasoning. Rerun the practice with different inputs after studying the solutions.

In [ ]:
from collections import Counter
related = sorted((r for r in records if r["path"] == "/rules"), key=lambda r:r["time"])
for r in related: print(r)
successful = [r for r in related if r["method"] == "DELETE" and 200 <= r["status"] < 300]
assert len(successful) == 1
incident = successful[0]
print("Likely causal request:", incident)
print("All 404s:", Counter(r["ip"] for r in records if r["status"] == 404))
print("Rules 404s:", Counter(r["ip"] for r in related if r["status"] == 404))
matching = [a for a in audit if a["time"] == incident["time"] and a["ip"] == incident["ip"]]
print("Audit match:", matching)

**Answers:** The successful DELETE is at 10:01:14 from 10.0.0.44 using account `admin`; its status is 204. The guest received 403. Request `r19` carries fragment `TRACE`. Counting errors alone does not establish causation. To identify a person, look for authenticated session records, device ownership, and possible shared or stolen credentials. A shared request ID would provide stronger correlation than time/IP alone.